# Stage 7.1 — Freeze and commit (the test set is NOT used here)

This notebook only *reads* `final_config.yaml` (written by `src/final_eval.freeze()` from development data) and checks that no
test prediction exists. The final run (`final_eval.run(confirm_final=True)`) is Prompt 7.2 and waits for the researcher.

### Step 1 — What was frozen

**WHAT:** load the config and summarise every frozen model: configuration (tuned / default), seeds, frozen CSI threshold (mean
over seeds), frozen trees/epochs and smearing factors.

**WHY:** everything that could still be "chosen" — model set, hyper-parameters, thresholds, number of trees/epochs, smearing,
back-transformation, seeds, statistical tests — is fixed now, from development data only, so nothing can be adapted to the
test years later.

**Look for:** 5 baselines, 22 classical entries (10 per task + CatBoost Tweedie + hurdle), 6 deep entries (3 models × 2 tasks),
thresholds between 0.3 and 0.6, inner slice 2017-01-01..2019-12-30.

In [1]:
import sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import final_eval as FE
cfg = FE.load_config()
print("pre-test commit:", cfg["freeze"]["pre_test_commit"], "| frozen on", cfg["freeze"]["date"])
print("development targets:", cfg["data"]["development"]["target_dates"], "| final training targets:",
      cfg["data"]["final_training_rows"]["target_dates"])
print("inner split:", cfg["data"]["inner_split_for_frozen_values"]["inner_train_target_dates"], "/ slice",
      cfg["data"]["inner_split_for_frozen_values"]["slice_target_dates"])
print("test targets (locked):", cfg["data"]["test"]["target_dates"], "| expected rows:", cfg["data"]["test"]["expected_n_rows"])
rows = []
for task, d in cfg["models"]["baselines"].items():
    for n, e in d.items():
        rows.append({"family": "baseline", "task": task, "model": n, "config": "Stage 3", "seeds": "0",
                     "threshold": e.get("threshold")})
for family in ("classical", "deep"):
    for task, d in cfg["models"][family].items():
        for n, e in d.items():
            fr = pd.DataFrame(e["frozen"]).T
            rows.append({"family": family, "task": task, "model": n, "config": e["config"] + (f" ({e['route']})" if "route" in e else ""),
                         "seeds": ",".join(map(str, e["seeds"])),
                         "threshold": fr["threshold"].mean() if "threshold" in fr else np.nan,
                         "threshold range": f"{fr['threshold'].min():.2f}-{fr['threshold'].max():.2f}" if "threshold" in fr else "",
                         "trees/epochs": ",".join(str(int(v)) for v in fr.get("n_trees", fr.get("best_epoch", pd.Series(dtype=float))).dropna())
                         if ("n_trees" in fr or "best_epoch" in fr) else "",
                         "smear": fr["smear_factor"].mean() if "smear_factor" in fr else np.nan})
frozen = pd.DataFrame(rows)
frozen.to_csv(ROOT / "reports/tables/07_frozen_models.csv", index=False)
with pd.option_context("display.max_rows", 60, "display.width", 220, "display.max_colwidth", 40):
    display(frozen.round(3))

pre-test commit: 827eae8e6320d1e620a9882bfe11acf9552d31d9 | frozen on 2026-10-17
development targets: ['1981-01-31', '2019-12-31'] | final training targets: ['1981-01-31', '2019-12-30']
inner split: ['1981-01-31', '2016-12-30'] / slice ['2017-01-01', '2019-12-30']
test targets (locked): ['2020-01-01', '2025-12-31'] | expected rows: 2192


/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,family,task,model,config,seeds,threshold,threshold range,trees/epochs,smear
0,baseline,cls,majority,Stage 3,0,0.500,NaN,NaN,NaN
1,baseline,cls,persistence,Stage 3,0,0.150,NaN,NaN,NaN
2,baseline,cls,climatology,Stage 3,0,0.470,NaN,NaN,NaN
3,baseline,cls,seasonal,Stage 3,0,0.450,NaN,NaN,NaN
4,baseline,cls,seasonal+persistence,Stage 3,0,0.370,NaN,NaN,NaN
5,baseline,reg,majority,Stage 3,0,NaN,NaN,NaN,NaN
6,baseline,reg,persistence,Stage 3,0,NaN,NaN,NaN,NaN
7,baseline,reg,climatology,Stage 3,0,NaN,NaN,NaN,NaN
8,baseline,reg,seasonal,Stage 3,0,NaN,NaN,NaN,NaN
9,baseline,reg,seasonal+persistence,Stage 3,0,NaN,NaN,NaN,NaN


**Interpretation:** see the summary in `DECISIONS.md` (7.1–7.3). The frozen numbers come from one inner fit per model and seed on
targets ≤ 2016-12-30 with the 2017–2019 slice — the same rule used in every CV fold (Decision 4.1), applied to the final training set.

**Paper note:** *All hyper-parameters, decision thresholds, early-stopping iteration counts and smearing factors were fixed on the
development data and committed before the test period was accessed.*

### Step 2 — Verify that no test prediction exists and the lock still holds

**WHAT:** list `models/test/`; check that `final_eval.run()` refuses without `confirm_final=True`; run `tests/test_final_eval.py`.

**Look for:** no parquet files and no manifest in `models/test/`; `PermissionError`; all tests pass.

In [2]:
files = sorted(p.name for p in FE.TEST_DIR.glob("*")) if FE.TEST_DIR.exists() else []
print("models/test contents:", files)
print("test prediction files:", [f for f in files if f.endswith(".parquet")], "| manifest:", "manifest.json" in files)
try:
    FE.run()
except PermissionError as e:
    print("lock holds ->", e)
r = subprocess.run([sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "-q", "tests/test_final_eval.py"],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])

models/test contents: ['README.md']
test prediction files: [] | manifest: False
lock holds -> Final evaluation is locked. Call run(confirm_final=True) only after the researcher's explicit Stage-7.2 confirmation.


14 passed, 4 warnings in 23.46s


### Step 3 — What counts as test-set contamination, and why the freeze commit protects the paper

**Contamination = any way in which information from the 2020–2025 test period influences a modelling decision.** Examples:
- **Direct use:** fitting, early-stopping, calibrating or choosing a threshold on test rows; fitting scalers/imputers on them.
- **Selection after looking:** running several configurations (or seeds, thresholds, routes, feature sets) on the test set and
  reporting the best one; dropping a model because it did badly on test; re-running "with a small fix" after seeing test scores.
- **Indirect leakage:** test-period statistics in features (e.g. a climatology or normalisation that includes 2020–2025);
  using the test years to decide *which* analyses or comparisons to report (garden of forking paths); changing the
  statistical test, the reference model or the multiplicity correction after the scores are known.
- **Inspection:** plotting or summarising 2020–2025 rainfall/targets before the final run (EDA was restricted to 1981–2019,
  Decision 1.4; integrity checks only).

**Not contamination:** using the observed weather of day t inside the test period as the *input* for the forecast of day t+1 —
that is exactly what an operational forecaster would have (rule 5).

**Why commit first.** The commit fixes, with a timestamp and hash, *what* will be evaluated and *how* it will be judged before
any test number exists. The final run records the commit it ran from and refuses to run if `final_config.yaml` or `src/` have
uncommitted changes or if the data differ from the frozen hashes. A reviewer can therefore verify that the reported test scores
are the one-shot outcome of a pre-registered protocol — the test estimate stays an unbiased estimate of performance on new years,
instead of the maximum of many tries (winner's curse). If anything must change after 7.2, it is reported as a deviation, with
both the frozen and the changed results.

**Paper note:** *The final configuration and evaluation code were committed (commit hash reported) before the 2020–2025 test period
was accessed, and the test set was evaluated exactly once.*